# Explicit SVD experiments

This notebook evaluates and tunes the explicit-feedback SVD retriever.

The experiments focus on both standalone candidate retrieval quality and the retriever's contribution to the full hybrid candidate pool. Hyperparameters are selected primarily using validation data, while the final test split remains untouched.

## Goals

- evaluate the minimum item-support threshold;
- study latent factor dimensionality;
- tune optimization-related hyperparameters;
- measure the effect of SVD changes on hybrid candidate recall;
- select a final SVD configuration for subsequent system experiments.

In [1]:
import os
from pathlib import Path

import pandas as pd

from shiki_recsys.config.settings import get_settings
from shiki_recsys.config.training import load_training_config
from shiki_recsys.database.session import (
    create_database_engine,
    create_session_factory,
)
from shiki_recsys.evaluation.retrievers import (
    RetrieverCandidateRun,
    evaluate_retriever_run,
    generate_retriever_evaluation_run,
)
from shiki_recsys.preprocessing.interactions import select_explicit_interactions
from shiki_recsys.preprocessing.known_items import build_known_items
from shiki_recsys.preprocessing.splitting import chronological_split
from shiki_recsys.retrievers.common import RetrieverName
from shiki_recsys.retrievers.explicit_svd import ExplicitSVDRetriever
from shiki_recsys.training.data import load_training_data
from shiki_recsys.training.retrievers import fit_retrievers

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

## Evaluation setup

The chronological split and candidate-generation pipeline are reused from the production code.

SVD variants are trained on the outer training split and evaluated against positive interactions from the validation split. The other retrievers remain fixed at their baseline configuration so that changes in hybrid candidate quality can be attributed to the SVD variant being tested.

The test split is not used during hyperparameter selection.

In [2]:
config = load_training_config(PROJECT_ROOT / "configs" / "training.yaml")
settings = get_settings()

engine = create_database_engine(settings)
session_factory = create_session_factory(engine)

try:
    with session_factory() as session:
        data = load_training_data(session)
finally:
    engine.dispose()

interactions = data.interactions
catalog = data.catalog

split_config = config.dataset.split

split = chronological_split(
    interactions,
    validation_fraction=split_config.validation_fraction,
    test_fraction=split_config.test_fraction,
    min_interactions_per_user=split_config.min_interactions_per_user,
)

pd.Series(
    {
        "train_interactions": len(split.train),
        "validation_interactions": len(split.validation),
        "test_interactions": len(split.test),
        "catalog_items": len(catalog),
        "retrieval_k": config.candidate_generation.retrieval_k,
        "positive_rating_threshold": (config.target.positive_rating_threshold),
    },
    name="value",
)

train_interactions           875697.0
validation_interactions      108915.0
test_interactions            108915.0
catalog_items                 23802.0
retrieval_k                     100.0
positive_rating_threshold         8.0
Name: value, dtype: float64

In [3]:
(
    popularity,
    explicit_svd,
    implicit_als,
    content_tfidf,
) = fit_retrievers(
    interactions=split.train,
    catalog=catalog,
    config=config,
)

In [4]:
candidate_count = config.candidate_generation.retrieval_k

baseline_run = generate_retriever_evaluation_run(
    history_interactions=split.train,
    target_interactions=split.validation,
    positive_rating_threshold=(config.target.positive_rating_threshold),
    candidate_count=candidate_count,
    popularity=popularity,
    explicit_svd=explicit_svd,
    implicit_als=implicit_als,
    content_tfidf=content_tfidf,
)

## Hyperparameter experiments

The following experiments evaluate the main SVD hyperparameters on the validation split.

Each configuration is assessed both as a standalone retriever and as a component of the hybrid candidate generator. The primary selection criteria are hybrid candidate recall, marginal recall contribution, and the number of uniquely retrieved relevant user-item pairs.

Only one parameter is varied at a time unless stated otherwise.

### Minimum item support

`min_item_explicit_ratings` controls which items are included in SVD training based on the number of available explicit ratings.

A lower threshold increases catalog support and allows the model to represent less frequently rated items, while a higher threshold restricts training to items with more reliable interaction histories.

The remaining SVD hyperparameters are kept at their baseline values during this experiment.

In [5]:
explicit_train = select_explicit_interactions(split.train)
svd_config = config.retrievers.explicit_svd
known_items_by_user = build_known_items(split.train)

min_rating_values = [1, 3, 5, 10, 20, 30, 50, 100]

svd_min_rating_results = []

for min_item_explicit_ratings in min_rating_values:
    retriever = ExplicitSVDRetriever(
        min_item_explicit_ratings=min_item_explicit_ratings,
        n_factors=svd_config.n_factors,
        n_epochs=svd_config.n_epochs,
        biased=svd_config.biased,
        learning_rate=svd_config.learning_rate,
        regularization=svd_config.regularization,
        init_mean=svd_config.init_mean,
        init_std_dev=svd_config.init_std_dev,
        random_state=config.random_seed,
    ).fit(explicit_train)

    candidates_by_user = {}

    for user_id, baseline_candidates in baseline_run.candidates_by_user.items():
        svd_candidates = retriever.retrieve(
            user_id=user_id,
            candidate_count=candidate_count,
            exclude_anime_ids=known_items_by_user.get(user_id, set()),
        )

        candidates_by_user[user_id] = {
            **baseline_candidates,
            RetrieverName.EXPLICIT_SVD: svd_candidates,
        }

    run = RetrieverCandidateRun(
        positive_items_by_user=baseline_run.positive_items_by_user,
        candidates_by_user=candidates_by_user,
    )

    evaluation = evaluate_retriever_run(
        run,
        candidate_count=candidate_count,
        catalog_size=len(catalog),
    )

    svd_metrics = (
        evaluation.individual_metrics
        .set_index("source")
        .loc[RetrieverName.EXPLICIT_SVD.value]
    )

    svd_contribution = next(
        contribution
        for contribution in evaluation.contributions
        if contribution.source == RetrieverName.EXPLICIT_SVD
    )

    svd_min_rating_results.append(
        {
            "min_item_explicit_ratings": min_item_explicit_ratings,
            "supported_items": len(retriever.supported_anime_ids),
            "svd_recall_at_100": svd_metrics["recall_at_k"],
            "svd_catalog_coverage_at_100": (
                svd_metrics["catalog_coverage_at_k"]
            ),
            "union_recall": evaluation.union_recall,
            "svd_marginal_recall": (
                svd_contribution.marginal_recall
            ),
            "svd_unique_relevant_hits": (
                svd_contribution.unique_relevant_hits
            ),
        }
    )

svd_min_rating_results = pd.DataFrame(svd_min_rating_results)

svd_min_rating_results

,min_item_explicit_ratings,supported_items,svd_recall_at_100,svd_catalog_coverage_at_100,union_recall,svd_marginal_recall,svd_unique_relevant_hits
0,1,16345,0.206848,0.090245,0.542120,0.064734,1440
1,3,11963,0.205329,0.089866,0.543561,0.066175,1482
2,5,9056,0.212937,0.087262,0.543957,0.066571,1493
3,10,6390,0.220364,0.086043,0.541332,0.063946,1443
4,20,4349,0.232922,0.079909,0.542727,0.065341,1472
5,30,3377,0.233079,0.075876,0.535813,0.058427,1383
6,50,2446,0.244276,0.067473,0.536222,0.058836,1370
7,100,1367,0.260461,0.049786,0.527026,0.049640,1075


#### Selection

Increasing the minimum number of explicit ratings per item improves standalone SVD Recall@100, reaching 0.260 at a threshold of 100. However, this improvement comes from restricting the model to a much smaller set of frequently rated items.

The hybrid-system results show the opposite trend. A threshold of 5 provides the highest candidate-pool recall (0.543), the highest SVD marginal recall contribution (0.0655), and the largest number of uniquely retrieved relevant user-item pairs (1,484).

Higher thresholds therefore make SVD stronger as an isolated retriever but less complementary to the other candidate generators. The baseline value of 5 is retained for subsequent experiments.

### Latent factor dimensionality

The number of latent factors controls the capacity of the SVD model. Too few factors may limit the model's ability to represent user preferences, while increasing the dimensionality also increases model size and computation and may provide diminishing retrieval gains.

The item-support threshold is fixed at 5, selected in the previous experiment. Other SVD parameters remain unchanged.

In [6]:
def evaluate_svd_variant(
    *,
    min_item_explicit_ratings: int = 5,
    n_factors: int = 64,
    n_epochs: int = 20,
    learning_rate: float | None = None,
    regularization: float = 0.10,
) -> dict[str, float | int]:
    """Evaluate one SVD configuration in the hybrid candidate pool."""
    retriever = ExplicitSVDRetriever(
        min_item_explicit_ratings=min_item_explicit_ratings,
        n_factors=n_factors or svd_config.n_factors,
        n_epochs=n_epochs or svd_config.n_epochs,
        biased=svd_config.biased,
        learning_rate=learning_rate or svd_config.learning_rate,
        regularization=(
            svd_config.regularization if regularization is None else regularization
        ),
        init_mean=svd_config.init_mean,
        init_std_dev=svd_config.init_std_dev,
        random_state=config.random_seed,
    ).fit(explicit_train)

    candidates_by_user = {}

    for user_id, baseline_candidates in baseline_run.candidates_by_user.items():
        svd_candidates = retriever.retrieve(
            user_id=user_id,
            candidate_count=candidate_count,
            exclude_anime_ids=known_items_by_user.get(user_id, set()),
        )

        candidates_by_user[user_id] = {
            **baseline_candidates,
            RetrieverName.EXPLICIT_SVD: svd_candidates,
        }

    run = RetrieverCandidateRun(
        positive_items_by_user=baseline_run.positive_items_by_user,
        candidates_by_user=candidates_by_user,
    )

    evaluation = evaluate_retriever_run(
        run,
        candidate_count=candidate_count,
        catalog_size=len(catalog),
    )

    svd_metrics = evaluation.individual_metrics.set_index("source").loc[
        RetrieverName.EXPLICIT_SVD.value
    ]

    contribution = next(
        item
        for item in evaluation.contributions
        if item.source == RetrieverName.EXPLICIT_SVD
    )

    return {
        "supported_items": len(retriever.supported_anime_ids),
        "svd_recall_at_100": svd_metrics["recall_at_k"],
        "svd_catalog_coverage_at_100": (svd_metrics["catalog_coverage_at_k"]),
        "union_recall": evaluation.union_recall,
        "svd_marginal_recall": contribution.marginal_recall,
        "svd_unique_relevant_hits": contribution.unique_relevant_hits,
    }

In [7]:
factor_values = [16, 32, 64, 96, 128, 192, 256, 384]

svd_factor_results = []

for n_factors in factor_values:
    result = evaluate_svd_variant(
        n_factors=n_factors,
    )

    svd_factor_results.append(
        {
            "n_factors": n_factors,
            **result,
        }
    )

svd_factor_results = pd.DataFrame(svd_factor_results)

svd_factor_results

,n_factors,supported_items,svd_recall_at_100,svd_catalog_coverage_at_100,union_recall,svd_marginal_recall,svd_unique_relevant_hits
0,16,9056,0.197446,0.088690,0.538948,0.061562,1416
1,32,9056,0.201704,0.089656,0.541385,0.063999,1467
2,64,9056,0.212937,0.087262,0.543957,0.066571,1493
3,96,9056,0.212739,0.086463,0.542524,0.065138,1498
4,128,9056,0.217370,0.084363,0.543595,0.066209,1514
5,192,9056,0.220986,0.083901,0.544444,0.067058,1514
6,256,9056,0.222442,0.082304,0.544407,0.067021,1548
7,384,9056,0.226013,0.079909,0.544905,0.067518,1542


#### Selection

Increasing the number of latent factors improves standalone SVD Recall@100, but the gain translates only weakly to the hybrid candidate pool.

Doubling the dimensionality from 64 to 128 factors increases SVD Recall@100 from 0.2114 to 0.2172, while union recall improves only from 0.5429 to 0.5436. The number of uniquely retrieved relevant user-item pairs also changes only slightly, from 1,484 to 1,492.

Higher-dimensional variants continue to improve standalone SVD retrieval, but the hybrid gains remain small and non-monotonic. Since model size and SVD computation grow approximately linearly with the number of factors, the additional complexity is not justified by the observed candidate-pool improvement.

The 64-factor configuration is therefore retained as the current efficiency-quality trade-off. This should not be interpreted as a proven global optimum; a higher-dimensional SVD may still be reconsidered later if it produces a meaningful improvement in end-to-end ranking metrics such as NDCG@20.

### Training epochs

The number of training epochs determines how long the SVD parameters are optimized on the training interactions.

Too few epochs may leave the model underfitted, while additional epochs increase training cost and may provide diminishing or negative validation gains.

The item-support threshold is fixed at 5 and the latent dimensionality at 64, based on the previous experiments. All other SVD parameters remain unchanged.

In [8]:
epoch_values = [5, 10, 15, 20, 30, 40]

svd_epoch_results = []

for n_epochs in epoch_values:
    result = evaluate_svd_variant(
        n_epochs=n_epochs,
    )

    svd_epoch_results.append(
        {
            "n_epochs": n_epochs,
            **result,
        }
    )

svd_epoch_results = pd.DataFrame(svd_epoch_results)

svd_epoch_results

,n_epochs,supported_items,svd_recall_at_100,svd_catalog_coverage_at_100,union_recall,svd_marginal_recall,svd_unique_relevant_hits
0,5,9056,0.224407,0.036888,0.538686,0.061300,1336
1,10,9056,0.216165,0.061255,0.541230,0.063844,1424
2,15,9056,0.211365,0.078229,0.542878,0.065492,1484
3,20,9056,0.212937,0.087262,0.543957,0.066571,1493
4,30,9056,0.209158,0.098773,0.542190,0.064804,1485
5,40,9056,0.210187,0.105705,0.542232,0.064846,1499


#### Selection

The number of training epochs has a substantial effect on the type of candidates produced by SVD. With only 5 epochs, standalone Recall@100 is highest, but catalog coverage and the retriever's contribution to the hybrid candidate pool are considerably lower.

Increasing the training duration improves coverage and hybrid contribution up to 20 epochs. At this point, union recall reaches 0.5440 and SVD marginal recall reaches 0.0666, both the highest values among the tested configurations.

Further training to 30 or 40 epochs increases catalog coverage but does not improve hybrid candidate recall. Since additional epochs affect offline training cost but not model size or inference complexity, 20 epochs are retained as the current training-duration trade-off.

This value is selected under the current learning-rate and regularization settings and may be reconsidered after those parameters are tuned.

### Regularization

Regularization controls the strength of the penalty applied to the learned latent factors and biases.

Too little regularization may allow the model to overfit the training interactions, while excessive regularization can suppress useful preference structure. The effect is evaluated using both standalone SVD retrieval quality and contribution to the hybrid candidate pool.

The item-support threshold is fixed at 5, latent dimensionality at 64, and training duration at 20 epochs. The learning rate remains at its baseline value.

In [9]:
regularization_values = [0.02, 0.05, 0.10, 0.20, 0.40]

svd_regularization_results = []

for regularization in regularization_values:
    result = evaluate_svd_variant(
        regularization=regularization,
    )

    svd_regularization_results.append(
        {
            "regularization": regularization,
            **result,
        }
    )

svd_regularization_results = pd.DataFrame(
    svd_regularization_results
)

svd_regularization_results

,regularization,supported_items,svd_recall_at_100,svd_catalog_coverage_at_100,union_recall,svd_marginal_recall,svd_unique_relevant_hits
0,0.02,9056,0.173232,0.153432,0.527882,0.050496,1171
1,0.05,9056,0.196209,0.134148,0.538158,0.060772,1360
2,0.10,9056,0.212937,0.087262,0.543957,0.066571,1493
3,0.20,9056,0.196615,0.035291,0.540331,0.062945,1392
4,0.40,9056,0.193545,0.019746,0.540946,0.063560,1342


#### Selection

Regularization has a clear effect on both standalone SVD retrieval and its contribution to the hybrid candidate pool.

Weak regularization increases catalog coverage substantially, but produces lower recall and fewer uniquely retrieved relevant user-item pairs. Stronger regularization reduces coverage sharply and also weakens the hybrid contribution.

A regularization strength of 0.10 provides the highest standalone Recall@100, union recall, marginal recall contribution, and number of unique relevant hits among the tested values.

Therefore, `regularization = 0.10` is retained for subsequent experiments.

### Learning rate

The learning rate controls the magnitude of parameter updates during SVD optimization.

A learning rate that is too small may result in insufficient optimization within the fixed number of epochs, while an excessively large value may lead to unstable updates or poorer generalization.

The item-support threshold is fixed at 5, latent dimensionality at 64, training duration at 20 epochs, and regularization at 0.10 based on the previous experiments.

In [10]:
learning_rate_values = [0.003, 0.005, 0.010, 0.015, 0.020, 0.030]

svd_learning_rate_results = []

for learning_rate in learning_rate_values:
    result = evaluate_svd_variant(
        learning_rate=learning_rate,
    )

    svd_learning_rate_results.append(
        {
            "learning_rate": learning_rate,
            **result,
        }
    )

svd_learning_rate_results = pd.DataFrame(
    svd_learning_rate_results
)

svd_learning_rate_results

,learning_rate,supported_items,svd_recall_at_100,svd_catalog_coverage_at_100,union_recall,svd_marginal_recall,svd_unique_relevant_hits
0,0.003,9056,0.227602,0.036846,0.532327,0.054941,1190
1,0.005,9056,0.225543,0.051634,0.536294,0.058908,1278
2,0.010,9056,0.216217,0.076170,0.542042,0.064656,1460
3,0.015,9056,0.212937,0.087262,0.543957,0.066571,1493
4,0.020,9056,0.206534,0.094740,0.542555,0.065169,1497
5,0.030,9056,0.203635,0.104781,0.543298,0.065911,1472


#### Selection

The learning rate substantially affects the balance between standalone SVD retrieval and complementary candidate generation.

Lower learning rates produce higher standalone Recall@100 but substantially lower catalog coverage and weaker contribution to the hybrid candidate pool. Increasing the learning rate improves candidate diversity, with hybrid performance peaking at 0.015.

At `learning_rate = 0.015`, union recall reaches 0.5440 and SVD marginal recall reaches 0.0666, both the highest values among the tested configurations. A slightly higher learning rate of 0.020 produces four additional unique relevant hits, but this does not compensate for its lower overall hybrid recall.

Therefore, `learning_rate = 0.015` is retained.

## Conclusion

The experiments evaluated the main hyperparameters of the explicit-feedback SVD retriever using the validation period, while keeping the final test split untouched.

The selected configuration is:

- `min_item_explicit_ratings = 5`
- `n_factors = 64`
- `n_epochs = 20`
- `learning_rate = 0.015`
- `regularization = 0.10`

The minimum item-support threshold of 5 provides the strongest contribution to the hybrid candidate pool despite higher thresholds improving standalone SVD recall.

Increasing the latent dimensionality beyond 64 factors improves standalone retrieval, but the corresponding gains in hybrid candidate recall are too small to justify the additional model size and computation. Therefore, 64 factors are retained as an efficiency-quality trade-off rather than as a proven global optimum.

Increasing the training duration from 15 to 20 epochs improves hybrid candidate recall without affecting model size or inference cost. Further training provides no additional hybrid recall gain.

The baseline learning rate and regularization values remain the strongest among the tested alternatives.

Compared with the original SVD configuration, the selected configuration therefore changes only the training duration from 15 to 20 epochs. The final SVD configuration will still be evaluated as part of the complete recommendation pipeline, where end-to-end ranking metrics such as NDCG@20 remain the primary measure of recommendation quality.